**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 14 — Human-in-the-Loop (HITL)

Three-tier routing:
- **Auto-approve** (P(hate) < 0.30): model confident → non-hateful
- **Human review** (0.30 ≤ P(hate) ≤ 0.75): uncertain → flag for annotation
- **Auto-remove** (P(hate) > 0.75): model confident → hateful

Interactive ipywidgets annotation UI + audit log.

In [ ]:
import os
import json
import re
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
DEV_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["dev"])
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEV_PATH is None:
    raise FileNotFoundError(f"Expected dev split under {DATA_DIR}")

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
DEV_PATH = str(DEV_PATH)
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Device         : {DEVICE}")

APPROVE_THRESHOLD = 0.30
REMOVE_THRESHOLD = 0.75


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


def clean_text(text):
    if not isinstance(text, str):
        return "[no text]"
    return re.sub(
        r"\s+",
        " ",
        re.sub(r"@\w+", " ", re.sub(r"https?://\S+", " ", text)),
    ).strip() or "[no text]"


dev_df = load_jsonl(DEV_PATH)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)
print(f"Dev: {len(dev_df)} samples")

In [ ]:
from pathlib import Path


def resolve_image_path(data_dir, image_ref):
    data_dir = Path(data_dir)
    image_ref = Path(str(image_ref))

    candidates = []
    if image_ref.is_absolute():
        candidates.append(image_ref)

    candidates.extend([
        data_dir / image_ref,
        data_dir.parent / image_ref,
    ])

    if image_ref.parts:
        if image_ref.parts[0] in {"img", "images"} and len(image_ref.parts) > 1:
            stripped = Path(*image_ref.parts[1:])
            candidates.extend([
                data_dir / stripped,
                data_dir.parent / stripped,
            ])
        elif image_ref.parts[0] not in {"img", "images"}:
            candidates.extend([
                data_dir / "img" / image_ref,
                data_dir / "images" / image_ref,
                data_dir.parent / "img" / image_ref,
                data_dir.parent / "images" / image_ref,
            ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image '{image_ref}' relative to {data_dir}")

In [ ]:
#  Load model 
class MemeDataset(Dataset):
    def __init__(self, df, data_dir, processor):
        self.df=df.reset_index(drop=True); self.data_dir=data_dir; self.processor=processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        try: img=Image.open(resolve_image_path(self.data_dir, row["img"])).convert("RGB")
        except: img=Image.new("RGB",(224,224),128)
        enc=self.processor(text=[str(row.get("clean_text",row["text"]))],images=img,
                            return_tensors="pt",padding="max_length",max_length=77,truncation=True)
        lbl=int(row["label"]) if "label" in row else -1
        return {"pixel_values":enc["pixel_values"].squeeze(0),
                "input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "label":torch.tensor(lbl,dtype=torch.long)}

class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self,pv,ids,mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv),-1),
                F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,d=512,h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self,i,t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,pv,ids,mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = HatefulMemeClassifier().to(DEVICE)
ckpt      = os.path.join(OUTPUT_DIR, "cross_attention_best.pt")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("Checkpoint loaded.")
else:
    print("WARNING: using random weights.")
model.eval()

In [ ]:
# â”€â”€ Batch inference + routing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dev_ds     = MemeDataset(dev_df, DATA_DIR, processor)
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False, num_workers=2)

all_probs = []
with torch.no_grad():
    for batch in dev_loader:
        pv   = batch["pixel_values"].to(DEVICE)
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        logits,_,_ = model(pv, ids, mask)
        all_probs.extend(torch.softmax(logits,-1)[:,1].cpu().numpy())

dev_df["prob"] = all_probs
dev_df["tier"] = dev_df["prob"].apply(
    lambda p: "auto_approve" if p < APPROVE_THRESHOLD
              else ("auto_remove" if p > REMOVE_THRESHOLD
                    else "human_review")
)

tier_counts = dev_df["tier"].value_counts()
print("Routing results:")
for tier, cnt in tier_counts.items():
    pct = 100 * cnt / len(dev_df)
    print(f"  {tier:<15}: {cnt:4d}  ({pct:.1f}%)")

In [ ]:
# â”€â”€ Routing visualization â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Tier breakdown
tier_counts.plot(kind="bar", ax=axes[0], color=["#4CAF50","#2196F3","#F44336"][:len(tier_counts)],
                  edgecolor="white")
axes[0].set_title("Routing Tier Distribution", fontweight="bold")
axes[0].set_xticklabels(tier_counts.index, rotation=15)
axes[0].set_ylabel("Count")

# Probability histogram coloured by tier
for tier, color in [("auto_approve","#4CAF50"),("human_review","#2196F3"),("auto_remove","#F44336")]:
    data = dev_df[dev_df.tier == tier]["prob"]
    axes[1].hist(data, bins=20, alpha=0.6, color=color, label=tier, edgecolor="white")

axes[1].axvline(APPROVE_THRESHOLD, color="green", linestyle="--", linewidth=2)
axes[1].axvline(REMOVE_THRESHOLD,  color="red",   linestyle="--", linewidth=2)
axes[1].set_title("Confidence Distribution by Tier", fontweight="bold")
axes[1].set_xlabel("P(hate)"); axes[1].legend()

plt.suptitle(f"HITL Routing  |  approve<{APPROVE_THRESHOLD}  remove>{REMOVE_THRESHOLD}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hitl_routing.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# â”€â”€ Annotations datastore â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
audit_path = os.path.join(OUTPUT_DIR, "hitl_audit_log.jsonl")
annotated_ids = set()

if os.path.exists(audit_path):
    with open(audit_path) as f:
        for line in f:
            try: annotated_ids.add(json.loads(line)["id"])
            except: pass
    print(f"Loaded {len(annotated_ids)} existing annotations.")
else:
    print("Starting fresh annotation session.")

def save_annotation(sample_id, img_path, text, prob, model_pred, human_label, note=""):
    entry = {
        "id": sample_id,
        "img": img_path,
        "text": text,
        "model_prob": float(prob),
        "model_pred": int(model_pred),
        "human_label": human_label,
        "annotator_note": note,
        "timestamp": datetime.datetime.utcnow().isoformat(),
    }
    with open(audit_path, "a") as f:
        f.write(json.dumps(entry) + "\n")
    annotated_ids.add(sample_id)

In [ ]:
#  ipywidgets annotation UI 
review_queue = dev_df[dev_df.tier == "human_review"].reset_index(drop=True)
review_queue = review_queue[~review_queue["id"].astype(str).isin(annotated_ids)]
print(f"Items pending human review: {len(review_queue)}")

queue_idx = [0]  # mutable pointer

def render_current():
    clear_output(wait=True)
    if queue_idx[0] >= len(review_queue):
        print("All items reviewed!")
        return

    row = review_queue.iloc[queue_idx[0]]
    print(f"Item {queue_idx[0]+1}/{len(review_queue)}  |  ID: {row['id']}")
    print(f"Model confidence P(hate) = {row.prob:.3f}")
    print(f"Text: {row['text']}")

    # Display image
    try:
        img = Image.open(resolve_image_path(DATA_DIR, row["img"]))
        buf = BytesIO()
        img.thumbnail((300, 300))
        img.save(buf, format="PNG")
        img_widget = widgets.Image(value=buf.getvalue(), format="png", width=300)
    except:
        img_widget = widgets.Label("Image not found")

    note_box = widgets.Textarea(placeholder="Optional note", layout=widgets.Layout(width="350px", height="60px"))

    def on_click(label):
        def handler(btn):
            save_annotation(str(row["id"]), row["img"], row["text"], row["prob"],
                             1 if row.prob > 0.5 else 0, label, note_box.value)
            queue_idx[0] += 1
            render_current()
        return handler

    btn_hate    = widgets.Button(description="Hateful",     button_style="danger")
    btn_benign  = widgets.Button(description="Not Hateful", button_style="success")
    btn_unsure  = widgets.Button(description="Unsure",      button_style="warning")
    btn_hate.on_click(on_click(1))
    btn_benign.on_click(on_click(0))
    btn_unsure.on_click(on_click(-1))

    progress = widgets.IntProgress(value=queue_idx[0], min=0, max=len(review_queue),
                                   description="Progress:", style={"bar_color": "#2196F3"})

    display(widgets.VBox([
        progress,
        img_widget,
        note_box,
        widgets.HBox([btn_hate, btn_benign, btn_unsure])
    ]))

# Launch
render_current()

In [ ]:
# â”€â”€ Audit log analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if os.path.exists(audit_path):
    audit_df = pd.read_json(audit_path, lines=True)
    print(f"Total annotations: {len(audit_df)}")
    print()

    # Agreement with model
    valid = audit_df[audit_df.human_label.isin([0, 1])].copy()
    if len(valid) > 0:
        valid["model_label"] = (valid["model_prob"] > 0.5).astype(int)
        agreement = (valid.human_label == valid.model_label).mean()
        print(f"Human-model agreement: {agreement:.1%}")

        model_overflags = ((valid.model_label==1) & (valid.human_label==0)).sum()
        model_underflags = ((valid.model_label==0) & (valid.human_label==1)).sum()
        print(f"Model over-flagged (FP): {model_overflags}")
        print(f"Model under-flagged (FN): {model_underflags}")

    unsure_pct = (audit_df.human_label == -1).mean()
    print(f"Annotator uncertainty rate: {unsure_pct:.1%}")
else:
    print("No annotations saved yet â€” run the UI cells above.")

In [ ]:
# â”€â”€ HITL summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("=" * 55)
print("HITL ROUTING SUMMARY")
print("=" * 55)
print(f"Thresholds: approve < {APPROVE_THRESHOLD} | remove > {REMOVE_THRESHOLD}")
print()
print("Routing impact:")
for tier, cnt in tier_counts.items():
    print(f"  {tier:<15}: {cnt:4d}  ({100*cnt/len(dev_df):.1f}%)")
print()
print("Design rationale:")
print("  - High-confidence non-hate flagged automatically by model at <30%")
print("  - High-confidence hate removed automatically at >75%")
print("  - Uncertain zone 30-75% routed to human reviewer")
print("  - All human decisions logged with timestamp for accountability")
print("  - Audit log can be used to fine-tune model on edge cases")
print("\nProceed to notebook 14 (Packaging & API).")